In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 5-point Generator

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

plt.rcParams["figure.figsize"] = (7, 5)

Device: cpu


In [ ]:
@dataclass
class StructuredGrid2D:
    nx: int
    ny: int
    lx: float
    ly: float
    thickness: float

    def __post_init__(self):
        self.n_cells = self.nx * self.ny
        self.dx = self.lx / self.nx
        self.dy = self.ly / self.ny
        self.bulk_volume = self.dx * self.dy * self.thickness
        self.face_i, self.face_j, self.face_area, self.face_distance = self._build_connectivity()
        self.cell_centers = self._build_centers()

    def cell_index(self, i, j):
        return i + self.nx * j

    def _build_centers(self):
        centers = np.zeros((self.n_cells, 2))
        for j in range(self.ny):
            for i in range(self.nx):
                c = self.cell_index(i, j)
                centers[c, 0] = (i + 0.5) * self.dx
                centers[c, 1] = (j + 0.5) * self.dy
        return centers

    def _build_connectivity(self):
        fi, fj, area, dist = [], [], [], []
        for j in range(self.ny):
            for i in range(self.nx):
                c = self.cell_index(i, j)
                if i + 1 < self.nx:
                    fi.append(c)
                    fj.append(self.cell_index(i + 1, j))
                    area.append(self.dy * self.thickness)
                    dist.append(self.dx)
                if j + 1 < self.ny:
                    fi.append(c)
                    fj.append(self.cell_index(i, j + 1))
                    area.append(self.dx * self.thickness)
                    dist.append(self.dy)
        return np.array(fi), np.array(fj), np.array(area, float), np.array(dist, float)

    def reshape(self, field):
        return np.asarray(field).reshape(self.ny, self.nx)

In [ ]:
@dataclass
class CoreyFluid:
    mu_w: float = 1e-3
    mu_o: float = 4e-3
    s_wr: float = 0.20
    s_or: float = 0.25
    krw0: float = 0.30
    kro0: float = 1.00
    n_w: float = 2.0
    n_o: float = 2.0

    @property
    def s_min(self):
        return self.s_wr

    @property
    def s_max(self):
        return 1.0 - self.s_or

    def effective_saturation(self, sw):
        swe = (sw - self.s_wr) / (1.0 - self.s_wr - self.s_or)
        return np.clip(swe, 0.0, 1.0)

    def mobilities(self, sw):
        swe = self.effective_saturation(sw)
        krw = self.krw0 * swe ** self.n_w
        kro = self.kro0 * (1.0 - swe) ** self.n_o
        lam_w = krw / self.mu_w
        lam_o = kro / self.mu_o
        lam_t = lam_w + lam_o
        fw = lam_w / (lam_t + 1e-30)
        return lam_w, lam_o, lam_t, fw


class IMPES2D:
    def __init__(self, grid, phi=0.01, perm_mD=1.0):
        self.grid = grid
        self.phi = phi
        self.perm = perm_mD * 9.869233e-16
        self.fi = grid.face_i
        self.fj = grid.face_j
        self.T_geom = self.perm * grid.face_area / grid.face_distance

        self.rows_pattern = np.concatenate([self.fi, self.fj, np.arange(grid.n_cells)])
        self.cols_pattern = np.concatenate([self.fj, self.fi, np.arange(grid.n_cells)])

        self.inj = grid.cell_index(0, 0)
        self.prod = grid.cell_index(grid.nx - 1, grid.ny - 1)

        year = 365.0 * 24.0 * 3600.0
        pore_volume = phi * grid.lx * grid.ly * grid.thickness
        self.q_inj = 0.1 * pore_volume / year

        self.q_total = np.zeros(grid.n_cells)
        self.q_total[self.inj] = self.q_inj
        self.q_total[self.prod] = -self.q_inj
        self.gauge_cell = self.prod

    def pressure_solve(self, sw, fluid):
        _, _, lam_t, fw = fluid.mobilities(sw)
        lt_i = lam_t[self.fi]
        lt_j = lam_t[self.fj]
        lt_face = 2.0 * lt_i * lt_j / (lt_i + lt_j + 1e-30)
        T_face = self.T_geom * lt_face

        diag = np.zeros(self.grid.n_cells)
        np.add.at(diag, self.fi, T_face)
        np.add.at(diag, self.fj, T_face)

        data = np.concatenate([-T_face, -T_face, diag])
        A = sp.csr_matrix((data, (self.rows_pattern, self.cols_pattern)), shape=(self.grid.n_cells, self.grid.n_cells)).tolil()
        b = self.q_total.copy()

        # Gauge condition for no-flow pressure equation
        A[self.gauge_cell, :] = 0.0
        A[self.gauge_cell, self.gauge_cell] = 1.0
        b[self.gauge_cell] = 0.0

        p = spla.spsolve(A.tocsc(), b)
        F_total = -T_face * (p[self.fj] - p[self.fi])
        return p, F_total, fw

    def cfl_dt(self, F_total, cfl=0.5):
        outflux = np.zeros(self.grid.n_cells)
        np.add.at(outflux, self.fi, np.maximum(F_total, 0.0))
        np.add.at(outflux, self.fj, np.maximum(-F_total, 0.0))
        outflux += np.maximum(-self.q_total, 0.0)
        pv = self.phi * self.grid.bulk_volume
        return float(np.min(cfl * pv / np.maximum(outflux, 1e-30)))

    def transport_step(self, sw, F_total, fw, fluid, dt):
        upwind = np.where(F_total >= 0.0, self.fi, self.fj)
        Fw = fw[upwind] * F_total

        div = np.zeros(self.grid.n_cells)
        np.add.at(div, self.fi, Fw)
        np.add.at(div, self.fj, -Fw)

        q_w = np.zeros(self.grid.n_cells)
        q_w[self.inj] = self.q_inj
        q_w[self.prod] = fw[self.prod] * (-self.q_inj)

        pv = self.phi * self.grid.bulk_volume
        sw_new = sw + dt / pv * (q_w - div)
        return np.clip(sw_new, fluid.s_min, fluid.s_max)

    def run(self, n_w=2.0, n_o=2.0, final_years=20.0, n_save=41, cfl=0.5, max_dt_days=30.0):
        year = 365.0 * 24.0 * 3600.0
        final_time = final_years * year
        save_times = np.linspace(0.0, final_time, n_save)

        fluid = CoreyFluid(n_w=n_w, n_o=n_o)
        sw = np.full(self.grid.n_cells, fluid.s_min)

        saved_t, saved_sw, saved_p = [], [], []

        t = 0.0
        next_save = 0

        while next_save < len(save_times):
            target = save_times[next_save]
            while t < target - 1e-9:
                p, F, fw = self.pressure_solve(sw, fluid)
                dt = self.cfl_dt(F, cfl=cfl)
                dt = min(dt, max_dt_days * 86400.0, target - t)
                sw = self.transport_step(sw, F, fw, fluid, dt)
                t += dt

            p, F, fw = self.pressure_solve(sw, fluid)
            saved_t.append(t)
            saved_sw.append(sw.copy())
            saved_p.append(p.copy())
            next_save += 1

        return {
            "times": np.asarray(saved_t),
            "saturation": np.asarray(saved_sw),
            "pressure": np.asarray(saved_p),
            "n_w": n_w,
            "n_o": n_o,
        }

In [ ]:
grid = StructuredGrid2D(nx=30, ny=30, lx=500.0, ly=500.0, thickness=50.0)
sim = IMPES2D(grid, phi=0.01, perm_mD=1.0)

param_train = [
    (2.0, 2.0),
    (2.0, 3.0),
    (3.0, 2.0),
    (3.0, 3.0),
    (4.0, 2.5),
    (2.5, 4.0),
]

solutions = []
for nw, no in param_train:
    print(f"Generating FV solution for n_w={nw}, n_o={no}")
    solutions.append(sim.run(n_w=nw, n_o=no, final_years=20.0, n_save=41))

print("Data generation finished.")

Generating FV solution for n_w=2.0, n_o=2.0
Generating FV solution for n_w=2.0, n_o=3.0
Generating FV solution for n_w=3.0, n_o=2.0
Generating FV solution for n_w=3.0, n_o=3.0
Generating FV solution for n_w=4.0, n_o=2.5
Generating FV solution for n_w=2.5, n_o=4.0
Data generation finished.
Dataset: (150000, 5) (150000, 2)
Pressure scale: 347715328.58612955


In [ ]:

def build_dataset(solutions, grid, max_points_per_solution=25000):
    X_list, Y_list = [], []
    centers = grid.cell_centers
    x = centers[:, 0] / grid.lx
    y = centers[:, 1] / grid.ly
    T_final = 20.0 * 365.0 * 24.0 * 3600.0

    global_p_scale = max(max(np.max(np.abs(sol["pressure"])) for sol in solutions), 1.0)

    for sol in solutions:
        times = sol["times"]
        Sw = sol["saturation"]
        P = sol["pressure"] / global_p_scale

        X_sol, Y_sol = [], []
        for it, t in enumerate(times):
            nc = grid.n_cells
            X = np.stack([
                x,
                y,
                np.full(nc, t / T_final),
                np.full(nc, sol["n_w"]),
                np.full(nc, sol["n_o"]),
            ], axis=1)
            Y = np.stack([Sw[it], P[it]], axis=1)
            X_sol.append(X)
            Y_sol.append(Y)

        X_sol = np.vstack(X_sol)
        Y_sol = np.vstack(Y_sol)

        if len(X_sol) > max_points_per_solution:
            idx = np.random.choice(len(X_sol), max_points_per_solution, replace=False)
            X_sol = X_sol[idx]
            Y_sol = Y_sol[idx]

        X_list.append(X_sol)
        Y_list.append(Y_sol)

    X = np.vstack(X_list).astype(np.float32)
    Y = np.vstack(Y_list).astype(np.float32)

    return X, Y, global_p_scale

X, Y, pressure_scale = build_dataset(solutions, grid)
print("Dataset:", X.shape, Y.shape)
print("Pressure scale:", pressure_scale)